# 08.10 · Sequences: RNNs, LSTMs and why Transformers replaced them

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/10-sequences-rnn-lstm/lesson.ipynb)

Read the [README](README.md) first and run the three RNN steps on paper. This notebook checks them, then shows a task where order is everything and a real one (film reviews) where it matters less than you would think.

Everything trains in seconds to a couple of minutes on a laptop CPU. Accuracies vary slightly by machine.

In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

Keras 3.15.1 · backend tensorflow


## 1. An RNN by hand

One number in, one number of memory. The same two weights are used at every step.

In [2]:
def rnn(xs, w_x=0.5, w_h=0.8, b=0.0):
    """A one-number RNN: h_t = tanh(w_x * x_t + w_h * h_(t-1) + b), starting from h_0 = 0."""
    h, states = 0.0, []
    for x in xs:
        h = np.tanh(w_x * x + w_h * h + b)
        states.append(round(float(h), 4))
    return states

print("x = [1, 0, -1]  →  h =", rnn([1, 0, -1]))
print("x = [-1, 0, 1]  →  h =", rnn([-1, 0, 1]))

x = [1, 0, -1]  →  h = [0.4621, 0.3537, -0.2137]
x = [-1, 0, 1]  →  h = [-0.4621, -0.3537, 0.2137]


Same three numbers, opposite order, opposite final memory: the RNN *knows the order*.

Keras' `SimpleRNN` computes exactly this when we give it the same weights:

In [3]:
cell = layers.SimpleRNN(1, activation="tanh", return_sequences=True)
cell.build((None, 3, 1))
cell.set_weights([np.array([[0.5]]), np.array([[0.8]]), np.array([0.0])])   # w_x, w_h, b
print(cell(np.array([1, 0, -1], dtype="float32").reshape(1, 3, 1)).numpy().ravel().round(4))

[ 0.4621  0.3537 -0.2137]


## 2. Vanishing memory

During training, the error signal travels back through every step and is multiplied by roughly the same factor each time. Multiply by less than 1 twenty times and almost nothing is left; by more than 1 and it explodes.

In [4]:
for factor in [0.5, 0.9, 1.0, 1.1]:
    print(f"{factor} ** 20 = {factor ** 20:.3g}")

0.5 ** 20 = 9.54e-07
0.9 ** 20 = 0.122
1.0 ** 20 = 1
1.1 ** 20 = 6.73


In [5]:
xs = [1] + [0] * 19            # one signal at step 1, then 19 steps of nothing
h = rnn(xs)
print("memory of the first input after steps 1, 5, 10, 20:", [h[i] for i in (0, 4, 9, 19)])

memory of the first input after steps 1, 5, 10, 20: [0.4621, 0.1719, 0.0555, 0.0059]


## 3. A task where order is everything

Each sequence has 20 tokens: random filler plus one `1` and one `2`. The label says whether the `1` comes first. Counting tokens tells you nothing; only order does.

In [6]:
def make_order_task(n, length=20, seed=0):
    """Sequences of random tokens 3-9 with one '1' and one '2'. Label 1 if '1' comes first."""
    rng = np.random.default_rng(seed)
    X = rng.integers(3, 10, (n, length))
    y = np.zeros(n, dtype=int)
    for i in range(n):
        p, q = rng.choice(length, 2, replace=False)
        X[i, p], X[i, q] = 1, 2
        y[i] = int(p < q)
    return X, y

def bag_model(length, vocab=10):
    """Averages the token embeddings: it cannot see order."""
    return keras.Sequential([keras.Input((length,)), layers.Embedding(vocab, 8),
                             layers.GlobalAveragePooling1D(),
                             layers.Dense(16, activation="relu"), layers.Dense(1, activation="sigmoid")])

def seq_model(length, cell=layers.LSTM, vocab=10):
    """Reads the tokens in order."""
    return keras.Sequential([keras.Input((length,)), layers.Embedding(vocab, 8),
                             cell(16), layers.Dense(1, activation="sigmoid")])

def fit_score(model, X, y, Xt, yt, epochs=15):
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    model.fit(X, y, epochs=epochs, batch_size=64, verbose=0)
    return model.evaluate(Xt, yt, verbose=0)[1]

X, y = make_order_task(4000, seed=0)
X_test, y_test = make_order_task(1000, seed=1)
print(X[0], "→ label", y[0])

[8 1 6 4 5 3 3 3 4 8 7 2 6 7 9 8 7 6 6 9] → label 1


In [7]:
keras.utils.set_random_seed(42)
acc_bag = fit_score(bag_model(20), X, y, X_test, y_test)
keras.utils.set_random_seed(42)
acc_lstm = fit_score(seq_model(20), X, y, X_test, y_test)
print(f"bag of tokens (ignores order): {acc_bag:.3f}")
print(f"LSTM (reads in order):         {acc_lstm:.3f}")

bag of tokens (ignores order): 0.521
LSTM (reads in order):         1.000


The bag model is at coin-flip level: it literally cannot see the answer. The LSTM reads the sequence in order and solves it.

## 4. Real text: IMDB film reviews

25,000 labelled reviews come with Keras, already turned into word numbers (we keep the 10,000 most common words and the last 200 words of each review). We use 10,000 for training and 5,000 for testing to keep it quick. In Module 09 you will work with ShopWave-style product reviews.

In [8]:
VOCAB, MAXLEN = 10_000, 200
(r_train, s_train), (r_test, s_test) = keras.datasets.imdb.load_data(num_words=VOCAB)
r_train = keras.utils.pad_sequences(r_train[:10_000], maxlen=MAXLEN); s_train = s_train[:10_000]
r_test = keras.utils.pad_sequences(r_test[:5_000], maxlen=MAXLEN);   s_test = s_test[:5_000]
print("train", r_train.shape, " test", r_test.shape, " share positive:", s_train.mean().round(3))

import time

def review_model(kind):
    core = {"bag": layers.GlobalAveragePooling1D(), "simplernn": layers.SimpleRNN(32),
            "lstm": layers.LSTM(32)}[kind]
    return keras.Sequential([keras.Input((MAXLEN,)), layers.Embedding(VOCAB, 32), core,
                             layers.Dense(1, activation="sigmoid")])

def fit_reviews(model, epochs):
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    start = time.time()
    model.fit(r_train, s_train, epochs=epochs, batch_size=64, validation_split=0.2, verbose=0)
    return model.evaluate(r_test, s_test, verbose=0)[1], time.time() - start

train (10000, 200)  test (5000, 200)  share positive: 0.505


In [9]:
review_results = {}
for kind, epochs in [("bag", 5), ("simplernn", 3), ("lstm", 3)]:
    keras.utils.set_random_seed(42)
    m = review_model(kind)
    acc, secs = fit_reviews(m, epochs)
    review_results[kind] = (m, acc)
    print(f"{kind:<10} {m.count_params():>8,} weights   test accuracy {acc:.3f}   ({secs:.0f} s)")

bag         320,033 weights   test accuracy 0.835   (1 s)


simplernn   322,113 weights   test accuracy 0.784   (4 s)


lstm        328,353 weights   test accuracy 0.836   (8 s)


The LSTM beats the plain RNN, but only matches the simple bag model. For deciding whether a review is positive, *which words appear* ("brilliant", "waste") carries most of the signal; word order adds little.

Test that directly: shuffle the words inside every test review and score again.

In [10]:
rng = np.random.default_rng(0)
shuffled = np.array([rng.permutation(r) for r in r_test])
for kind in ["bag", "lstm"]:
    m, acc = review_results[kind]
    acc_shuffled = m.evaluate(shuffled, s_test, verbose=0)[1]
    print(f"{kind:<5} normal order {acc:.3f}   words shuffled {acc_shuffled:.3f}")

bag   normal order 0.835   words shuffled 0.835


lstm  normal order 0.836   words shuffled 0.740


The bag model does not care about order at all (it averages). The LSTM drops when the order is destroyed, so it *does* use order, but on this task that knowledge did not make it better than ignoring order.

## Quiz answers

1. The hidden state $h_t$: a summary of everything the network has read so far.
2. The same $w_x$, $w_h$ and $b$ are used at every step.
3. $0.5^{20} \approx 9.5 \times 10^{-7}$: the learning signal from 20 steps back is almost zero, so the RNN cannot learn long-range patterns.
4. The forget gate (what to drop from memory), the input gate (what new information to store) and the output gate (what to reveal as $h_t$).
5. RNNs read one step at a time (slow, hard to parallelise) and struggle with long distances; Transformers look at all words at once with attention.

**Next:** [08.11 The same model in PyTorch](../11-pytorch/) (optional), or the [project](../project/).